Más análisis exploratorio de los datos...

In [63]:
import pandas as pd 
import numpy as np

In [64]:
catalogo_productos = pd.read_csv("Datos-limpiados/catalogo_productos.csv")
especificaciones_cajas = pd.read_csv("Datos-limpiados/especificaciones_cajas.csv")
operaciones_planta = pd.read_csv("Datos-limpiados/operaciones_planta.csv")
procurement_cajas = pd.read_csv("Datos-limpiados/procurement_cajas.csv")

Agregamos las cargas máximas para cada tipo de caja (pasamos de mm a m)

In [ ]:
#Calculo el perímetro. Divido por 1000 para pasar a m
perimetro_cajas = 2 * (especificaciones_cajas["caja_exterior_largo"] + especificaciones_cajas["caja_exterior_ancho"])/ 1000 

ect_por_grosor = {2.5: 600, 2.7: 730, 3.0: 1000, 4.1: 1200, 4.5: 1400, 4.6: 1450, 4.7: 1500, 4.8: 1550, 5.0: 1650}

ects = [ect_por_grosor[g] for g in especificaciones_cajas['caja_grosor_mm']]

especificaciones_cajas['carga_max'] = ects * perimetro_cajas / 9.81

Averigüemos si la carga soportada actualmente se acerca a la máxima

¿cuenta el peso del producto en la base de la pila?

In [ ]:
merged = catalogo_productos.merge(
    especificaciones_cajas[['caja_tipo_id', 'cantidad_cajas_alto', 'carga_max',
                            'caja_exterior_largo','caja_exterior_ancho','caja_exterior_alto',
                            'caja_interior_largo','caja_interior_ancho','caja_interior_alto']],
    on='caja_tipo_id',
    how='left'
)
#El -1 es para NO contar la caja de la base
catalogo_productos['carga_soportada'] = merged['peso_neto_caja'] * (merged['cantidad_cajas_alto'] - 1)

In [166]:
catalogo_productos[catalogo_productos['carga_soportada']/ merged['carga_max'] > 1]

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,peso_neto_paquete,caja_tipo_id,carga_soportada


El numero máximo de cajas que se pueden apilar para un producto dado es

$$n_{max\_carga} = \lfloor(carga\_max/peso\_neto\_paquete)\rfloor + 1 $$

mientras este número no haga que la pila sopbrepase los 1800 mm

In [156]:
merged[merged['caja_exterior_alto'] * merged['cantidad_cajas_alto'] >= 1800]

,codigo_producto,descripcion_producto,ingrediente_forma,tipo_proyecto,subcategoria,categoria,tamaño_corte,peso_neto_caja,tamaño_paquete,cantidad_paquetes,...,caja_tipo_id,carga_soportada,cantidad_cajas_alto,carga_max,caja_exterior_largo,caja_exterior_ancho,caja_exterior_alto,caja_interior_largo,caja_interior_ancho,caja_interior_alto


In [157]:
merged['n_por_ancho_pallet'] = (1200 // merged['caja_exterior_ancho']).astype(int)   # 1200mm / ancho caja
merged['n_por_largo_pallet'] = (800 // merged['caja_exterior_largo']).astype(int)   # 800mm / largo caja
merged['n_capas_max_altura'] = (1800 // merged['caja_exterior_alto']).astype(int) # limitado por altura

In [158]:
merged['n_max_carga'] = ((merged['carga_max'] // merged['peso_neto_caja']) + 1).astype(int)

El n_max de cajas que se pueden apilar es el mínimo entre el número máximo que soporta el tipo de caja y el máximo número antes de que sobrepase los 1800 mm de altura

In [159]:
merged['n_max'] = np.minimum(
    merged['n_capas_max_altura'],
    merged['n_max_carga']
)

In [160]:
merged[merged['n_capas_max_altura'] <= merged['n_max_carga']].shape

(435, 26)

Siempre gana la altura! Se puede bajar la carga_max entonces... probemos con el mínimor grosor permitido = 3.0 mm

In [161]:
nuevo_per = 2 * (merged["caja_interior_largo"] + merged["caja_interior_ancho"] + 12.0)/ 1000
merged['max_carga_3mm'] = 1000 * nuevo_per / 9.81
merged['n_max_carga_3mm'] = ((merged['max_carga_3mm'] // merged['peso_neto_caja']) + 1).astype(int)

In [162]:
merged['n_capas_max_altura_3mm'] = (1800 // (merged['caja_interior_alto'] + 6.0)).astype(int)

In [163]:
merged[merged['n_capas_max_altura_3mm'] <= merged['n_max_carga_3mm']].shape

(435, 29)

Sigue ganando altura!

In [164]:
merged[merged['n_capas_max_altura_3mm'] > merged['n_capas_max_altura']].shape

(63, 29)

Además en algunos casos ganamos una capa más

In [167]:
merged['n_por_ancho_pallet_3mm'] = (1200 // (merged['caja_interior_ancho'] + 6.0)).astype(int)
merged['n_por_largo_pallet_3mm'] = (800 // (merged['caja_interior_largo'] + 6.0)).astype(int)

In [170]:
merged[merged['n_por_ancho_pallet_3mm'] == merged['n_por_ancho_pallet']].shape

(435, 31)

In [176]:
merged[merged['n_por_largo_pallet_3mm'] == merged['n_por_largo_pallet']].shape

(430, 31)

In [177]:
ids = merged[merged['n_por_largo_pallet_3mm'] < merged['n_por_largo_pallet']]['caja_tipo_id']
print(ids.shape)
especificaciones_cajas[especificaciones_cajas['caja_tipo_id'].isin(ids)]['caja_grosor_mm'].value_counts()

(5,)


caja_grosor_mm
2.5    2
Name: count, dtype: int64

En ancho no ganamos nada... en largo tampoco y de hecho "perdimos" aunque en realidad esos 5 son con grosor menor el cual no está permitido